# EvalHub SDK Demo

This notebook demonstrates the EvalHub Python SDK for programmatic LLM evaluation.

EvalHub orchestrates evaluation jobs using providers like lm-evaluation-harness, Garak, GuideLLM, and LightEval,
with results automatically tracked in MLflow.

**Prerequisites:**
- EvalHub deployed in your namespace (run `deploy.sh` first)
- A model deployed via InferenceService
- `disableLMEval: false` in OdhDashboardConfig

In [ ]:
# Install the EvalHub SDK
!pip install -q "eval-hub-sdk[client]"

In [ ]:
import os
import json
import time

# --- Configuration ---
# Set these to match your deployment
EVALHUB_URL = os.environ.get("EVALHUB_URL", "http://evalhub.redhat-ods-applications.svc:8080")
NAMESPACE = os.environ.get("NAMESPACE", "lmeval-demo")
TENANT = NAMESPACE

# Model endpoint (auto-detect or set manually)
MODEL_NAME = os.environ.get("MODEL_NAME", "")  # e.g. "Qwen3-4B"
MODEL_URL = os.environ.get("MODEL_URL", "")     # e.g. "https://model-endpoint/v1"

print(f"EvalHub URL: {EVALHUB_URL}")
print(f"Namespace/Tenant: {TENANT}")

## 1. Connect to EvalHub

In [ ]:
from evalhub import EvalHubClient

client = EvalHubClient(
    base_url=EVALHUB_URL,
    tenant=TENANT,
)

# Check health
health = client.health()
print(f"Status: {health['status']}")
print(f"Version: {health['version']}")
print(f"Active evaluations: {health['active_evaluations']}")

## 2. Discover Providers and Benchmarks

In [ ]:
# List available providers
providers = client.list_providers()
print("Available providers:")
for p in providers:
    print(f"  - {p['name']}: {len(p.get('benchmarks', []))} benchmarks")

In [ ]:
# List benchmarks for lm-evaluation-harness
benchmarks = client.list_benchmarks(provider="lm_evaluation_harness")
print(f"\nlm-evaluation-harness benchmarks ({len(benchmarks)}):")
for b in benchmarks[:15]:
    category = b.get('category', 'general')
    print(f"  [{category}] {b['name']}: {b.get('description', '')}")
if len(benchmarks) > 15:
    print(f"  ... and {len(benchmarks) - 15} more")

In [ ]:
# List available collections (pre-built benchmark suites)
collections = client.list_collections()
print("Available collections:")
for c in collections:
    n_benchmarks = len(c.get('benchmarks', []))
    print(f"  - {c['name']}: {n_benchmarks} benchmarks")
    for b in c.get('benchmarks', [])[:5]:
        print(f"      {b['provider']}/{b['name']}")

## 3. Auto-Detect Model Endpoint

In [ ]:
import subprocess

# Env vars injected via notebook-env ConfigMap; fallback for standalone use
if not MODEL_NAME or not MODEL_URL:
    MODEL_URL = os.environ.get("BASE_URL", "")
    detected = ""
    detected_ns = ""

    if not MODEL_NAME or not MODEL_URL:
        try:
            result = subprocess.run(
                ["oc", "get", "llminferenceservice", "-A", "--no-headers",
                 "-o", "custom-columns=NS:.metadata.namespace,NAME:.metadata.name"],
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0 and result.stdout.strip():
                parts = result.stdout.strip().split("\n")[0].split()
                detected_ns, detected = parts[0], parts[1]
        except Exception:
            pass

        if detected:
            MODEL_NAME = MODEL_NAME or detected
            if not MODEL_URL:
                MODEL_URL = f"https://{detected}-kserve-workload-svc.{detected_ns}.svc:8000/v1"

if not MODEL_NAME:
    MODEL_NAME = input("Enter model name: ")
if not MODEL_URL:
    MODEL_URL = input("Enter model URL (e.g. https://model-svc.ns.svc:8000/v1): ")

print(f"Model: {MODEL_NAME}")
print(f"URL: {MODEL_URL}")

## 4. Submit an Evaluation Job

Run a single benchmark (MMLU) against the model.

In [ ]:
# Get a ServiceAccount token for model authentication
# Prefer long-lived Secret-bound token (doesn't expire), fall back to short-lived
import base64 as _b64
SA_TOKEN = ""
try:
    secret_result = subprocess.run(
        ["oc", "get", "secret", "lmeval-sa-token", "-n", NAMESPACE,
         "-o", "jsonpath={.data.token}"],
        capture_output=True, text=True, timeout=10
    )
    if secret_result.stdout.strip():
        SA_TOKEN = _b64.b64decode(secret_result.stdout.strip()).decode()
        print("Using long-lived SA token from Secret")
except Exception:
    pass

if not SA_TOKEN:
    try:
        token_result = subprocess.run(
            ["oc", "create", "token", "lmeval-sa", "-n", NAMESPACE, "--duration=8h"],
            capture_output=True, text=True, timeout=10
        )
        SA_TOKEN = token_result.stdout.strip()
        print("Using short-lived SA token (8h)")
    except Exception:
        SA_TOKEN = ""
        print("Warning: Could not get SA token. Set SA_TOKEN manually.")

# Submit MMLU evaluation
job = client.submit_job(
    model={
        "name": MODEL_NAME,
        "url": MODEL_URL,
        "type": "openai-chat",
        "token": SA_TOKEN,
    },
    benchmarks=[
        {
            "provider": "lm_evaluation_harness",
            "name": "mmlu",
            "args": {"num_fewshot": 5},
        }
    ],
    experiment={
        "name": f"evalhub-demo-{MODEL_NAME}",
    },
)

job_id = job["id"]
print(f"Job submitted: {job_id}")
print(f"Status: {job['status']}")

In [ ]:
# Poll for job completion
print(f"Waiting for job {job_id}...")
while True:
    status = client.get_job(job_id)
    state = status["status"]
    print(f"  Status: {state}", end="\r")
    if state in ("completed", "failed", "cancelled", "partially_failed"):
        break
    time.sleep(15)

print(f"\nFinal status: {state}")

## 5. Retrieve Results

In [ ]:
results = client.get_job_results(job_id)

print(f"Job: {job_id}")
print(f"Model: {MODEL_NAME}")
print(f"Status: {results.get('status', 'unknown')}")
print(f"\nBenchmark Results:")
print("-" * 60)

for benchmark in results.get("benchmarks", []):
    name = benchmark.get("name", "unknown")
    score = benchmark.get("score", "N/A")
    metric = benchmark.get("primary_metric", "")
    passed = benchmark.get("passed", None)
    status_icon = "PASS" if passed else ("FAIL" if passed is False else "--")
    print(f"  {name}: {score:.4f} ({metric}) [{status_icon}]")

## 6. Run a Collection (Benchmark Suite)

Collections group benchmarks from multiple providers into reusable evaluation suites.

In [ ]:
# Submit a safety evaluation using the safety-and-fairness collection
safety_job = client.submit_job(
    model={
        "name": MODEL_NAME,
        "url": MODEL_URL,
        "type": "openai-chat",
        "token": SA_TOKEN,
    },
    collection="safety-and-fairness-v1",
    experiment={
        "name": f"safety-eval-{MODEL_NAME}",
    },
)

print(f"Safety evaluation job submitted: {safety_job['id']}")
print(f"Status: {safety_job['status']}")
print("\nThis runs multiple safety benchmarks -- check progress in the dashboard:")
print("  Develop and train > Evaluations")

## 7. List All Jobs

In [ ]:
jobs = client.list_jobs()
print(f"Total jobs: {len(jobs)}")
print(f"{'ID':<36} {'Model':<20} {'Status':<15} {'Benchmarks'}")
print("-" * 90)
for j in jobs:
    model = j.get('model', {}).get('name', 'unknown')
    n_benchmarks = len(j.get('benchmarks', []))
    collection = j.get('collection', '')
    bench_info = collection if collection else f"{n_benchmarks} benchmarks"
    print(f"{j['id']:<36} {model:<20} {j['status']:<15} {bench_info}")

## Summary

This notebook demonstrated:

1. **Connecting** to EvalHub via the Python SDK
2. **Discovering** available providers, benchmarks, and collections
3. **Submitting** individual benchmark evaluations
4. **Running** pre-built collection suites (safety, leaderboard)
5. **Retrieving** and displaying results

### Next Steps

- View results in MLflow: experiment tracking with metrics and artifacts
- Use the dashboard: **Develop and train > Evaluations** for visual job management
- CLI: `evalhub submit --model ... --benchmark ...` for terminal workflows
- Custom adapters: extend EvalHub with your own evaluation frameworks